# Dataset Evaluation: state estimation, anomaly detection, add-to-train comparison

## Paper configurations
1. State estimation: `ANOMALY_DATASET=False, TASKS=['peg_pick','probe','wrap']`
2. State estimation + extended train: same + `ADD_TO_TRAIN_CONFIGS` with injection entries
3. Anomaly detection: `ANOMALY_DATASET=True, TASKS=['peg_pick','probe','wrap']`
4. Growing-classes experiment: `ANOMALY_DATASET=False, TASKS=['additional']`

## Optional output flags
- `SHOW_CONFUSION_MATRICES` — inline confusion matrix per task split
- `SHOW_HISTOGRAMS` — per-model accuracy histogram (single dataset-config mode)

In [ ]:
import numpy as np
from tqdm import tqdm
from nocode_robot_programming.state_decision.utils import (
    kill_other_ipykernels, user_study_tasks_only, user_study_nice_model_names,
    y_cls_to_nice_name, user_study_plot_hist, set_seed,
)
from nocode_robot_programming.state_decision.state_decider import (
    StateDeciderBase, DINOFeaturePresence, DINOFeaturePresenceConcat,
    DINOFeaturePresenceAttnGated, DINOWithMIL, StateDeciderSIFT, AEGP,
)
from nocode_robot_programming.jupyter_plot import jupyter_plot as ipt, show_gray_video_cuda
from nocode_robot_programming.state_decision.plots.benchmark_plot import (
    visualize_accuracies, visualize_accuracies_3d,
)
from nocode_robot_programming.state_decision.plots import pretty_confusion_matrix
from nocode_robot_programming.state_decision_dataset_prepare.dataset_auto import (
    TrajectoryDatasetEvaluationViewBuilder,
)
kill_other_ipykernels(force=True)
set_seed()

# ── Task & dataset ──────────────────────────────────────────────────────────────
CRITERIA_CSV    = "trajectory_criteria.csv"
TASKS           = ['peg_pick', 'probe', 'wrap']  # or ['additional']
ANOMALY_DATASET = True                           # True → anomaly detection mode
OUT_DIR         = f'{"anomaly_detection" if ANOMALY_DATASET else "state_estimation"}'

# ── Models ──────────────────────────────────────────────────────────────────────
percentile_keep = 0.1 if ANOMALY_DATASET else None
DECIDERS = [
    DINOFeaturePresenceAttnGated(dino_variant="dinov2_vits14", percentile_keep=percentile_keep,
                                 attn_mode="hard", head_reduce="mean", attn_keep=0.4),
    DINOFeaturePresenceConcat(dino_variant="dinov2_vits14", percentile_keep=percentile_keep),
    # DINOFeaturePresence(dino_variant="dinov2_vits14", percentile_keep=percentile_keep),
    # DINOFeaturePresence(dino_variant="facebook/dinov3-vits16-pretrain-lvd1689m", percentile_keep=percentile_keep),
    # DINOWithMIL(dino_variant="dinov2_vits14", att_hidden=128, dropout_p=0.1, lr=1e-4, weight_decay=1e-3, epochs=1000),
    # StateDeciderSIFT(method="SIFT"),
    # AEGP(binary=False, pix=64),
]

# ── Training-data injection ─────────────────────────────────────────────────────
# Single entry → standard eval (2-D heatmap).  Multiple entries → injection sweep (3-D).
ADD_TO_TRAIN_CONFIGS = [
    ([], "no_inject"),
    # ([0, 1, 2],          "inject_1each"),
    # ([0, 1, 2, 0, 1, 2], "inject_2each"),
]

# ── Dataset-quality filters ─────────────────────────────────────────────────────
# Single entry → standard eval.  Multiple entries → grouped histogram comparison.
HEATMAP_VALID_CONFIGS = ["Nominal", "Visual distractor", "Limited observability"]
DATASET_CONFIGS = [
    {"name": "Nominal",                  "use_criteria": frozenset([""])},
    {"name": "Visual distractor",        "use_criteria": frozenset(["distractor"])},
    {"name": "Limited observability",    "use_criteria": frozenset(["target_features_not_visible_or_too_small",
                                                                      "hand_in_scene", "very_bad_train_data"])},
    {"name": "Acquisition/setup failure","use_criteria": frozenset(["camera_not_working",
                                                                      "teaching_multiple_things_at_once"])},
]

# ── Optional detailed output ────────────────────────────────────────────────────
SHOW_CONFUSION_MATRICES = False  # per-task confusion matrices inline
SHOW_HISTOGRAMS         = True  # per-model histogram (active when 1 dataset config)

In [ ]:
# all_viz[decider_name]["viz"][add_name] = {"train": [...], "test": [...], "names": [...]}
all_viz = {}

for decider in DECIDERS:
    viz = {add_name: {"train": [], "test": [], "names": []}
           for _, add_name in ADD_TO_TRAIN_CONFIGS}

    for add_to_train, add_name in ADD_TO_TRAIN_CONFIGS:
        grouped_stats = {}
        
        for dc in DATASET_CONFIGS:
            is_heatmap_config = dc['name'] in HEATMAP_VALID_CONFIGS
            dataset_builder = TrajectoryDatasetEvaluationViewBuilder(
                criteria_csv=CRITERIA_CSV,
                use_criteria=dc["use_criteria"],
            )
            stats = {}
            desc = f"{dc['name']} / {add_name} / {decider.short_name}"

            for task_name in tqdm(user_study_tasks_only(dataset_builder, TASKS), desc=desc):
                for d_train, d_test, d_text in dataset_builder.load_eval_from_task(
                        task_name, anomaly=ANOMALY_DATASET, add_to_train=add_to_train):
                    try:
                        decider.train(d_train.X, d_train.y_int, d_train.y_cls)
                        ipt.save()  # capture training-loss figure (if any)
                    except AssertionError:
                        continue

                    y_pred_train = decider.predict_many(d_train.X)
                    train_acc = (np.array(d_train.y_names) == np.array(y_pred_train)).mean()
                    pretty_confusion_matrix.pp_matrix_from_string_data(
                        d_train.y_names, y_pred_train,
                        name=f"train,{decider.short_name}", show=False); ipt.save()

                    y_pred_test = decider.predict_many(d_test.X)
                    test_acc = (np.array(d_test.y_names) == np.array(y_pred_test)).mean()
                    pretty_confusion_matrix.pp_matrix_from_string_data(
                        d_test.y_names, y_pred_test,
                        name=f"test,{decider.short_name}", show=False); ipt.save()

                    if SHOW_CONFUSION_MATRICES:
                        ipt.show(small=True)
                    else:
                        ipt.delete()

                    stats[d_text]        = test_acc
                    # Accumulate for heatmap using the first dataset config only
                    if is_heatmap_config:
                        viz[add_name]["train"].append(train_acc * 100)
                        viz[add_name]["test"].append(test_acc * 100)
                        viz[add_name]["names"].append(d_text.split(",")[0])

            grouped_stats[dc["name"]] = stats

        if SHOW_HISTOGRAMS:
            user_study_plot_hist(
                grouped_stats, savename=f"{add_name}_{decider.short_name}.pdf",
                bins=21, folder=OUT_DIR,
            )

    all_viz[decider.short_name] = viz

# Heatmap plots

In [ ]:
add_names  = [add_name for _, add_name in ADD_TO_TRAIN_CONFIGS]
task_names = next(iter(all_viz.values()))[add_names[0]]["names"]

if len(ADD_TO_TRAIN_CONFIGS) == 1:
    # ── Standard 2-D heatmap (single injection level — equivalent to 03) ───────
    train_2d = [all_viz[dec][add_names[0]]["train"] for dec in all_viz]
    test_2d  = [all_viz[dec][add_names[0]]["test"]  for dec in all_viz]
    visualize_accuracies(
        train_2d, test_2d,
        list(all_viz.keys()), task_names,
        out_dir=OUT_DIR, jupyter_plot=True,
    )
    all_train = [v for dec in all_viz.values() for v in dec[add_names[0]]["train"]]
    all_test  = [v for dec in all_viz.values() for v in dec[add_names[0]]["test"]]
    print(f"Overall  train: {np.mean(all_train):.1f}%   test: {np.mean(all_test):.1f}%")

else:
    # ── 3-D heatmap: cells = no-inj / +1-traj / +2-traj (equivalent to 04) ────
    if task_names:
        train_3d = [[[all_viz[dec][an]["train"][j] for an in add_names]
                     for j in range(len(task_names))]
                    for dec in all_viz]
        test_3d  = [[[all_viz[dec][an]["test"][j]  for an in add_names]
                     for j in range(len(task_names))]
                    for dec in all_viz]
        visualize_accuracies_3d(
            train_3d, test_3d,
            list(all_viz.keys()), task_names,
            out_dir=OUT_DIR, jupyter_plot=True,
        )
    for an in add_names:
        vals = [v for dec in all_viz.values() for v in all_viz[dec][an]["test"]]
        if vals:
            print(f"{an}: mean test = {np.mean(vals):.1f}%")

ipt.show()
